# ELECTRO-VOLT — 01. Exploración de Datos (EDA)
**Proyecto Integrador · Maestría en IA · UEES**

**Autores:** Jordan Vera Coello & Jorge Estrella Frere

En este notebook se realiza la auditoría de calidad de datos y el análisis exploratorio (EDA) de las mediciones de señales eléctricas trifásicas provenientes de `data/raw/classData.csv`:
1. Carga validada de datos mediante el pipeline modular `src/`.
2. Reporte de calidad de datos (ausencia de nulos y duplicados).
3. Inspección estadística y rangos físicos de corrientes y voltajes.
4. Distribución de las 6 clases originales de fallas.
5. Análisis de correlación entre señales de fase.
6. Detección y demostración del ordenamiento secuencial en bloques del dataset.


In [ ]:
import sys
from pathlib import Path

# Agregar raíz del proyecto al sys.path para importar src
raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.data_processing import cargar_datos, verificar_calidad_datos, mapear_clases
from src.utils import COLUMNAS, MAPA_6, ORDEN_6, SEMILLA

sns.set_theme(style="whitegrid")
print("[OK] Entorno y módulos modulares de src/ inicializados correctamente.")


## 1. Carga del Dataset
Cargamos el archivo `classData.csv` utilizando la función modular `cargar_datos()` de `src/data_processing.py`.


In [ ]:
df_crudo = cargar_datos()
print(f"Dimensiones del dataset: {df_crudo.shape[0]} filas × {df_crudo.shape[1]} columnas")
df_crudo.head()


## 2. Auditoría de Calidad de Datos
Verificamos valores nulos, registros duplicados y estadísticas descriptivas de las 6 señales continuas.


In [ ]:
calidad = verificar_calidad_datos(df_crudo)
print("=== REPORTE DE CALIDAD ===")
print(f"Valores nulos: {calidad['valores_nulos']}")
print(f"Filas duplicadas: {calidad['filas_duplicadas']}")
print(f"Total registros: {calidad['total_filas']}")

print("\n=== ESTADÍSTICAS DESCRIPTIVAS DE SEÑALES CONTINUAS ===")
df_crudo[COLUMNAS].describe().round(3)


## 3. Mapeo de Clases Originales (6 Clases)
Las banderas binarias `G, C, B, A` representan el estado de tierra y fases involucradas:
- `0000`: Normal (sin falla)
- `1001`: LG (Fase A a tierra)
- `0110`: LL (Línea a línea Fases B-C)
- `1011`: LLG (Doble línea a tierra Fases A-B)
- `0111`: LLL (Trifásica simétrica)
- `1111`: LLLG (Trifásica a tierra)


In [ ]:
df = mapear_clases(df_crudo, modo="6_clases")
conteo_6 = df["clase_6"].value_counts().reindex(ORDEN_6)
print("Distribución de 6 clases originales:\n", conteo_6)

plt.figure(figsize=(8, 4))
conteo_6.plot(kind="bar", color="steelblue")
plt.title("Muestras por Clase Original (6 Clases)", fontsize=13)
plt.ylabel("Cantidad de Muestras")
plt.xlabel("Tipo de Falla")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Matriz de correlación entre señales eléctricas
Evaluamos las interdependencias entre las tensiones (`Va, Vb, Vc`) y corrientes (`Ia, Ib, Ic`).


In [ ]:
plt.figure(figsize=(7, 5))
matriz_corr = df[COLUMNAS].corr()
sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Matriz de Correlación entre Señales Trifásicas", fontsize=13)
plt.tight_layout()
plt.show()


## 5. Alerta Crítica del Dataset: Bloques Secuenciales
El archivo viene ordenado cronológicamente por bloques contiguos de clases. Demostramos cuántas transiciones ocurren si no se mezclan los datos.


In [ ]:
bloques = int((df["clase_6"] != df["clase_6"].shift()).sum())
print(f"ALERTA: Alerta: El dataset viene agrupado en exactamente {bloques} bloques secuenciales contiguos.")
print("Por este motivo, TODO particionado (train_test_split, StratifiedKFold o learning_curve) DEBE aplicar obligatoriamente shuffle=True con SEMILLA=42.")
